# TT6581 - PDM External Filter Design

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sympy.solvers import solve
from sympy import Symbol, sqrt

## Poles and Response

In [ ]:
# 4th order Bessel poles
p1 = [1.3596, 0.4071] # [Re, Im] Conjugated
p2 = [0.9877, 1.2476] # [Re, Im] Conjugated

t = np.linspace(0, 2*np.pi, 100)
fig, ax = plt.subplots()
ax.axhline(0, color='black')
ax.axvline(0, color='black')
ax.plot(np.cos(t), np.sin(t), color='black', alpha=0.5)

for p in [p1, p2]:
    ax.plot(-p[0], p[1], 'o', color='black')
    ax.plot(-p[0], -p[1], 'o', color='black')
    ax.plot([0, -p[0]], [0, p[1]], color='black', linestyle='--', alpha=0.5)
    ax.plot([0, -p[0]], [0, -p[1]], color='black', linestyle='--', alpha=0.5)

ax.set_aspect('equal', 'box')
ax.set_xlabel('Re')
ax.set_ylabel('Im')
ax.set_title('4th Order Bessel LP')

## Quality Factor

In [ ]:
wn_1 = np.sqrt((p1[0]**2) + (p1[1]**2)) # Natural frequency, 1st stage
Q_1  = wn_1 / (2*p1[0])                 # Q-factor, 1st stage
wn_2 = np.sqrt((p2[0]**2) + (p2[1]**2)) # Natural frequency, 1st stage
Q_2  = wn_2 / (2*p2[0])                 # Q-factor, 1st stage

print('=== 1st Stage ===')
print(f'Natural frequency: {wn_1}')
print(f'Quality factor: {Q_1}')
print('\n=== 2nd Stage ===')
print(f'Natural frequency: {wn_2}')
print(f'Quality factor: {Q_2}')

## Sallen-Key Component Values

Letting $R_1 = mR$, $R_2 = R$, $C_1 = C$, $C_2 = nC$, and $K = 1$ results in:

$$
f_c = \frac{1}{2 * \pi * R C \sqrt{mn}}, \hspace{20pt} Q = \frac{\sqrt{mn}}{m + 1}
$$

This sets the gain to 0 dB in the pass band. Start the design by determining the ratios $m$ and $n$ for the required $Q$ of the filter, and then selecting $C$ and calculating $R$ to set $f_c$.

By choosing $m = 1$ (equal resistors), we have:

$$
Q = \frac{\sqrt{n}}{2} \hspace{10pt} \Rightarrow \hspace{10pt} n = 4Q^2
$$

In [ ]:
E24 = np.array([1.0, 1.1, 1.2, 1.3, 1.5, 1.6, 1.8, 2.0, 2.2, 2.4, 2.7, 3.0, 3.3, 3.6, 3.9, 4.3, 4.7, 5.1, 5.6, 6.2, 6.8, 7.5, 8.2, 9.1])
E96 = np.array([1.00, 1.02, 1.05, 1.07, 1.10, 1.13, 1.15, 1.18, 1.21, 1.24, 1.27, 1.30, 1.33, 1.37, 1.40, 1.43, 1.47, 1.50, 1.54, 1.58, 1.62, 1.65, 1.69, 1.74, 1.78, 1.82, 1.87, 1.91, 1.96, 2.00, 2.05, 2.10, 2.15, 2.21, 2.26, 2.32, 2.37, 2.43, 2.49, 2.55, 2.61, 2.67, 2.74, 2.80, 2.87, 2.94, 3.01, 3.09, 3.16, 3.24, 3.32, 3.40, 3.48, 3.57, 3.65, 3.74, 3.83, 3.92, 4.02, 4.12, 4.22, 4.32, 4.42, 4.53, 4.64, 4.75, 4.87, 4.99, 5.11, 5.23, 5.36, 5.49, 5.62, 5.76, 5.90, 6.04, 6.19, 6.34, 6.49, 6.65, 6.81, 6.98, 7.15, 7.32, 7.50, 7.68, 7.87, 8.06, 8.25, 8.45, 8.66, 8.87, 9.09, 9.31, 9.53, 9.76])

def snap_to_series(value, series):
    magnitude = 10**np.floor(np.log10(value))
    normalized = value / magnitude
    idx = (np.abs(series - normalized)).argmin()
    return series[idx] * magnitude

def get_real_world_components(fn_target, C_base, n_ideal):
    C2_real = snap_to_series(C_base, E24)
    C1_real = snap_to_series(C_base * n_ideal, E24)

    # Recalculate ideal Resistor with real capacitors
    R_ideal = 1 / (2 * np.pi * fn_target * np.sqrt(C1_real * C2_real))

    R_real = snap_to_series(R_ideal, E96)
    return R_real, C1_real, C2_real

## 1st Stage
Q   = Q_1   # Q-factor, 1st stage
fc  = 21e3  # [Hz]
C   = 1e-9  # [F]

m = 1 # Equal resistors
n = 4 * (Q**2)

fc = fc * wn_1

R1_real, C1_real, C2_real = get_real_world_components(fc, C, n)

print('=== 1st Stage ===')
print(f"R1 = R2 = {R1_real:.0f} Ohms (1% E96)")
print(f"C1 (Feedback) = {C1_real * 1e9:.1f} nF (5% E24)")
print(f"C2 (Ground) = {C2_real * 1e9:.1f} nF (5% E24)")

## 1st Stage
Q   = Q_2   # Q-factor, 1st stage
fc  = 21e3  # [Hz]
C   = 1e-9  # [F]

m = 1 # Equal resistors
n = 4 * (Q**2)

fc = fc * wn_2

R1_real, C1_real, C2_real = get_real_world_components(fc, C, n)

print('\n=== 2st Stage ===')
print(f"R1 = R2 = {R1_real:.0f} Ohms (1% E96)")
print(f"C1 (Feedback) = {C1_real * 1e9:.1f} nF (5% E24)")
print(f"C2 (Ground) = {C2_real * 1e9:.1f} nF (5% E24)")